# Vitara AI — NLP Stress/Emotion Model
**Architecture:** IndoBERT (`indobenchmark/indobert-base-p2`) + Multi-Task Head  
**Tasks:** Emotion Classification (5-class) + Stress Level Regression  
**Target:** Accuracy ≥ 85% | Stress MAE ≤ 0.02  
**Runtime:** T4 GPU (Google Colab)  
**Feature:** TensorBoard integration in custom training loop


## 1. Install Dependencies

In [5]:
# 1. Install dependensi
!pip install -q --upgrade transformers==4.44.2 datasets tokenizers tf-keras scikit-learn

print('✅ Dependensi terinstal.')

✅ Dependensi terinstal.


## 2. 🔧 Imports & Config

In [6]:
import os
# WAJIB diset sebelum import tensorflow atau transformers
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import time, datetime, json
import numpy as np
import pandas as pd
import tensorflow as tf

print(f'✅ TensorFlow Version: {tf.__version__}')

# Import HF
from transformers import TFBertModel, AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

print('✅ SUCCESS! TFBertModel berhasil dimuat.')

✅ TensorFlow Version: 2.21.0
✅ SUCCESS! TFBertModel berhasil dimuat.


In [7]:
# ── Hyperparameters ──────────────────────────────────────────────────────
MODEL_NAME     = 'indobenchmark/indobert-base-p2'
MAX_LEN        = 128
BATCH_SIZE     = 16
PHASE1_EPOCHS  = 5      # frozen BERT head warmup
PHASE2_EPOCHS  = 15     # full fine-tune
LR_PHASE1      = 3e-4   # head-only, can afford higher LR
LR_PHASE2      = 2e-5   # BERT fine-tune, must be small
WARMUP_RATIO   = 0.1
DROPOUT_RATE   = 0.2
LABEL_SMOOTHING = 0.1
CLIP_NORM      = 1.0    # gradient clipping
SEED           = 42

# GDrive paths  ← sesuaikan jika beda
GDRIVE_DATA_DIR = '/content/drive/MyDrive/vitara/data/nlp/raw/'
GDRIVE_OUT_DIR  = '/content/drive/MyDrive/vitara/models/nlp/'
GDRIVE_LOG_DIR  = '/content/drive/MyDrive/vitara/logs/nlp/'
GDRIVE_PROCESSED_DIR = '/content/drive/MyDrive/vitara/data/nlp/processed/'

EMOTION_LABELS = ['happy', 'sad', 'anxious', 'angry', 'neutral']
NUM_EMOTIONS   = len(EMOTION_LABELS)

tf.random.set_seed(SEED)
np.random.seed(SEED)
print('✅ Config ready')

✅ Config ready


## 3. Mount Google Drive & Load Dataset

In [8]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(GDRIVE_OUT_DIR, exist_ok=True)
os.makedirs(GDRIVE_LOG_DIR, exist_ok=True)
os.makedirs(GDRIVE_PROCESSED_DIR, exist_ok=True)
print('✅ Drive mounted')

Mounted at /content/drive
✅ Drive mounted


In [9]:
DATASET_FILENAME = 'journals (oversampling).csv'
print(f'Loading dataset: {DATASET_FILENAME}')

df = pd.read_csv(os.path.join(GDRIVE_DATA_DIR, DATASET_FILENAME))
assert all(c in df.columns for c in ['clean_text','emotion_label','stress_label']), \
    f'Missing columns: {df.columns.tolist()}'

df['emotion_label'] = df['emotion_label'].str.lower().str.strip()
df['stress_label']  = df['stress_label'].astype(np.float32)
df = df.dropna(subset=['clean_text','emotion_label','stress_label'])
df = df[df['emotion_label'].isin(EMOTION_LABELS)]

print(f'\nDataset: {len(df):,} rows')
print(df['emotion_label'].value_counts())
print(f"\nStress stats:\n{df['stress_label'].describe()}")
df.head(3)

Loading dataset: journals (oversampling).csv

Dataset: 10,004 rows
emotion_label
happy      2001
neutral    2001
sad        2001
anxious    2001
angry      2000
Name: count, dtype: int64

Stress stats:
count    10004.000000
mean         0.560073
std          0.305031
min          0.000000
25%          0.260000
50%          0.670000
75%          0.830000
max          1.000000
Name: stress_label, dtype: float64


,clean_text,emotion_label,stress_label
0,akhirnya lolos interview kerja tapi ya sudahla...,happy,0.13
1,ah jimin sudah sembuh senang banget,happy,0.04
2,kesal banget hari ini dari tadi kepikiran bare...,angry,0.91


## 4. Preprocess & Tokenize

In [10]:
label_encoder = LabelEncoder()
label_encoder.fit(EMOTION_LABELS)
y_emotion = label_encoder.transform(df['emotion_label']).astype(np.int32)
y_stress  = df['stress_label'].values.astype(np.float32)
texts     = df['clean_text'].tolist()

print('Class map:', dict(zip(label_encoder.classes_,
                              label_encoder.transform(label_encoder.classes_))))

# 70 / 15 / 15 split
X_tv, X_test, ye_tv, ye_test, ys_tv, ys_test = train_test_split(
    texts, y_emotion, y_stress, test_size=0.15, random_state=SEED, stratify=y_emotion)
X_train, X_val, ye_train, ye_val, ys_train, ys_val = train_test_split(
    X_tv, ye_tv, ys_tv, test_size=0.176, random_state=SEED, stratify=ye_tv)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

# ── Simpan dataset hasil split ke GDrive (processed) ─────────────────────
train_df = pd.DataFrame({
    'clean_text': X_train,
    'emotion_label': label_encoder.inverse_transform(ye_train),
    'stress_label': ys_train
})
val_df = pd.DataFrame({
    'clean_text': X_val,
    'emotion_label': label_encoder.inverse_transform(ye_val),
    'stress_label': ys_val
})
test_df = pd.DataFrame({
    'clean_text': X_test,
    'emotion_label': label_encoder.inverse_transform(ye_test),
    'stress_label': ys_test
})

train_df.to_csv(os.path.join(GDRIVE_PROCESSED_DIR, 'train.csv'), index=False)
val_df.to_csv(os.path.join(GDRIVE_PROCESSED_DIR, 'val.csv'), index=False)
test_df.to_csv(os.path.join(GDRIVE_PROCESSED_DIR, 'test.csv'), index=False)
print(f'✅ Split datasets berhasil disimpan ke: {GDRIVE_PROCESSED_DIR}')


Class map: {np.str_('angry'): np.int64(0), np.str_('anxious'): np.int64(1), np.str_('happy'): np.int64(2), np.str_('neutral'): np.int64(3), np.str_('sad'): np.int64(4)}
Train: 7,006 | Val: 1,497 | Test: 1,501
✅ Split datasets berhasil disimpan ke: /content/drive/MyDrive/vitara/data/nlp/processed/


In [11]:
print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts_list, max_len=MAX_LEN):
    enc = tokenizer(
        texts_list, max_length=max_len, padding='max_length',
        truncation=True, return_tensors='np'
    )
    return (enc['input_ids'].astype(np.int32),
            enc['attention_mask'].astype(np.int32),
            enc['token_type_ids'].astype(np.int32))

print('Tokenizing ...')
train_ids, train_mask, train_tt = tokenize(X_train)
val_ids,   val_mask,   val_tt   = tokenize(X_val)
test_ids,  test_mask,  test_tt  = tokenize(X_test)
print('✅ Done')

Loading tokenizer: indobenchmark/indobert-base-p2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Tokenizing ...
✅ Done


In [12]:
def make_dataset(ids, mask, tt, ye, ys, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((
        {'input_ids': ids, 'attention_mask': mask, 'token_type_ids': tt},
        {'emotion_output': ye, 'stress_output': ys}
    ))
    if shuffle:
        ds = ds.shuffle(4096, seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_ids, train_mask, train_tt, ye_train, ys_train, BATCH_SIZE, True)
val_ds   = make_dataset(val_ids,   val_mask,   val_tt,   ye_val,   ys_val,   BATCH_SIZE)
test_ds  = make_dataset(test_ids,  test_mask,  test_tt,  ye_test,  ys_test,  BATCH_SIZE)
print(f'Steps/epoch: {len(train_ds)}')

Steps/epoch: 438


## 5. Model Architecture

> BERT dibungkus sebagai subclass `tf.keras.layers.Layer`,  
> memastikan `bert.trainable` benar-benar mengontrol seluruh ~124M parameter.

In [13]:
class IndoBERTEncoder(tf.keras.layers.Layer):
    """
    Wrapper BERT sebagai Keras Layer proper.
    Memastikan bert.trainable = True/False mengontrol ~124M parameter.
    """
    def __init__(self, model_name, **kwargs):
        super().__init__(**kwargs)
        self.bert = TFBertModel.from_pretrained(model_name)

    def call(self, inputs, training=False):
        ids, mask, tt = inputs['input_ids'], inputs['attention_mask'], inputs['token_type_ids']
        out = self.bert(input_ids=ids, attention_mask=mask,
                        token_type_ids=tt, training=training)
        return out.last_hidden_state[:, 0, :]  # CLS token [batch, 768]


def build_model(dropout_rate=DROPOUT_RATE):
    # Inputs
    inp_ids  = tf.keras.Input((MAX_LEN,), dtype=tf.int32, name='input_ids')
    inp_mask = tf.keras.Input((MAX_LEN,), dtype=tf.int32, name='attention_mask')
    inp_tt   = tf.keras.Input((MAX_LEN,), dtype=tf.int32, name='token_type_ids')

    # BERT encoder
    bert_enc = IndoBERTEncoder(MODEL_NAME, name='indobert')
    cls = bert_enc({'input_ids': inp_ids,
                    'attention_mask': inp_mask,
                    'token_type_ids': inp_tt})
    cls = tf.keras.layers.Dropout(dropout_rate)(cls)

    # ── Emotion Head ──────────────────────────────────────────────────────
    e = tf.keras.layers.Dense(768, activation='gelu', name='emo_dense1')(cls)
    e = tf.keras.layers.Dropout(dropout_rate)(e)
    emotion_out = tf.keras.layers.Dense(
        NUM_EMOTIONS, activation='softmax', name='emotion_output')(e)

    # ── Stress Head ───────────────────────────────────────────────────────
    s = tf.keras.layers.Dense(256, activation='gelu', name='str_dense1')(cls)
    s = tf.keras.layers.BatchNormalization()(s)
    s = tf.keras.layers.Dropout(dropout_rate)(s)
    s = tf.keras.layers.Dense(64, activation='gelu', name='str_dense2')(s)
    stress_out = tf.keras.layers.Dense(
        1, activation='sigmoid', name='stress_output')(s)

    model = tf.keras.Model(
        inputs=[inp_ids, inp_mask, inp_tt],
        outputs=[emotion_out, stress_out]
    )
    return model


print(f'Loading {MODEL_NAME} ...')
model = build_model()

# Simpan referensi ke BERT layer (dipakai saat set trainable)
bert_layer = model.get_layer('indobert')
total_params = model.count_params()
print(f'Total params: {total_params:,}')
model.summary(line_length=80)

Loading indobenchmark/indobert-base-p2 ...


tf_model.h5:   0%|          | 0.00/656M [00:00<?, ?B/s]

Some layers from the model checkpoint at indobenchmark/indobert-base-p2 were not used when initializing TFBertModel: ['mlm___cls', 'nsp___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model checkpoint at indobenchmark/indobert-base-p2.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


Total params: 125,250,182
Model: "model"
________________________________________________________________________________
 Layer (type)           Output Shape            Param   Connected to            
                                                 #                              
 attention_mask (Input  [(None, 128)]           0       []                      
 Layer)                                                                         
                                                                                
 input_ids (InputLayer  [(None, 128)]           0       []                      
 )                                                                              
                                                                                
 token_type_ids (Input  [(None, 128)]           0       []                      
 Layer)                                                                         
                                                                    

## 6. Loss Functions & LR Scheduler

> Focal Loss + Huber + WarmupLinearDecay

In [14]:
class WeightedFocalLoss(tf.keras.losses.Loss):
    """
    Focal Loss + Label Smoothing untuk multi-class emotion.
    Handles class imbalance dan mengurangi overconfidence.
    """
    def __init__(self, gamma=2.0, smoothing=LABEL_SMOOTHING, **kwargs):
        super().__init__(**kwargs)
        self.gamma     = gamma
        self.smoothing = smoothing

    def call(self, y_true, y_pred):
        n = tf.cast(NUM_EMOTIONS, tf.float32)
        y_true_oh = tf.one_hot(tf.cast(tf.squeeze(y_true), tf.int32), depth=NUM_EMOTIONS)
        # Label smoothing
        y_smooth  = y_true_oh * (1.0 - self.smoothing) + self.smoothing / n
        y_pred    = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce        = -tf.reduce_sum(y_smooth * tf.math.log(y_pred), axis=-1)
        pt        = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
        focal_w   = tf.pow(1.0 - pt, self.gamma)
        return tf.reduce_mean(focal_w * ce)


class WarmupLinearDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps, total_steps):
        self.peak_lr, self.warmup, self.total = peak_lr, float(warmup_steps), float(total_steps)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * step / tf.maximum(self.warmup, 1.0)
        decay_lr  = self.peak_lr * tf.maximum(0.0, (self.total - step) / (self.total - self.warmup))
        return tf.cond(step < self.warmup, lambda: warmup_lr, lambda: decay_lr)

    def get_config(self):
        return {'peak_lr': self.peak_lr, 'warmup': self.warmup, 'total': self.total}


# Instantiate losses (dipakai di training loop)
focal_loss = WeightedFocalLoss(gamma=2.0, name='focal')
huber_loss = tf.keras.losses.Huber(delta=0.1, name='huber')

print('✅ Loss & scheduler defined')

✅ Loss & scheduler defined


## 7. Custom Training Loop — `tf.GradientTape`

> loop manual menggunakan `tf.GradientTape`.  
> Setiap iterasi: forward pass → hitung loss → backward → clip gradients → optimizer step.  
> Metrics (accuracy & MAE) dikelola secara eksplisit menggunakan `tf.keras.metrics`.

In [15]:
# ── Metric Objects (reset setiap epoch) ──────────────────────────────────
train_emo_acc  = tf.keras.metrics.SparseCategoricalAccuracy(name='train_emo_acc')
train_str_mae  = tf.keras.metrics.MeanAbsoluteError(name='train_str_mae')
val_emo_acc    = tf.keras.metrics.SparseCategoricalAccuracy(name='val_emo_acc')
val_str_mae    = tf.keras.metrics.MeanAbsoluteError(name='val_str_mae')



# ── TensorBoard Writers (Inisialisasi) ───────────────────────────────────
import datetime
current_time = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
train_log_dir = os.path.join(GDRIVE_LOG_DIR, current_time, 'train')
val_log_dir   = os.path.join(GDRIVE_LOG_DIR, current_time, 'validation')
train_summary_writer = tf.summary.create_file_writer(train_log_dir)
val_summary_writer   = tf.summary.create_file_writer(val_log_dir)
print(f'✅ TensorBoard logs will be saved to: {GDRIVE_LOG_DIR}{current_time}')


# ── Satu Training Step ────────────────────────────────────────────────────
@tf.function
def train_step(inputs, labels, optimizer):
    ye_true = labels['emotion_output']
    ys_true = tf.expand_dims(labels['stress_output'], -1)

    with tf.GradientTape() as tape:
        emo_pred, str_pred = model(
            [inputs['input_ids'], inputs['attention_mask'], inputs['token_type_ids']],
            training=True
        )
        loss_emo = focal_loss(ye_true, emo_pred)
        loss_str = huber_loss(ys_true, str_pred)
        total_loss = loss_emo + loss_str

    # Hitung gradients hanya terhadap variabel yang trainable
    grads = tape.gradient(total_loss, model.trainable_variables)

    # Clip gradients
    grads, _ = tf.clip_by_global_norm(grads, CLIP_NORM)

    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    # Update metrics
    train_emo_acc.update_state(ye_true, emo_pred)
    train_str_mae.update_state(ys_true, str_pred)
    return total_loss, loss_emo, loss_str


# ── Satu Validation Step ──────────────────────────────────────────────────
@tf.function
def val_step(inputs, labels):
    ye_true = labels['emotion_output']
    ys_true = tf.expand_dims(labels['stress_output'], -1)

    emo_pred, str_pred = model(
        [inputs['input_ids'], inputs['attention_mask'], inputs['token_type_ids']],
        training=False
    )
    val_emo_acc.update_state(ye_true, emo_pred)
    val_str_mae.update_state(ys_true, str_pred)


# ── Main Training Loop ────────────────────────────────────────────────────
def run_training_loop(phase_name, n_epochs, lr, bert_trainable,
                      history_store, best_ckpt_path=None, epoch_offset=0):
    # Set BERT trainable
    bert_layer.trainable = bert_trainable
    n_trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
    print(f'  BERT trainable={bert_trainable} | Trainable params: {n_trainable:,}')

    # Build optimizer dengan LR scheduler
    total_steps = len(train_ds) * n_epochs
    warmup      = int(total_steps * WARMUP_RATIO)
    sched       = WarmupLinearDecay(lr, warmup, total_steps)
    optimizer   = tf.keras.optimizers.AdamW(
        learning_rate=sched, weight_decay=0.01
    )

    # PERBAIKAN: Build optimizer secara eksplisit sebelum masuk ke tf.function
    # Ini mencegah error "tf.function only supports singleton tf.Variables"
    optimizer.build(model.trainable_variables)

    best_val_acc = -1.0
    train_start  = time.time()

    print('=' * 80)
    wib_tz = datetime.timezone(datetime.timedelta(hours=7))
    ts     = datetime.datetime.now(wib_tz).strftime('%Y-%m-%d %H:%M:%S WIB')
    print(f'🚀 [Vitara AI] {phase_name} Started at {ts}')
    print('=' * 80)

    for epoch in range(n_epochs):
        ep_start = time.time()

        # Reset semua metric di awal epoch
        train_emo_acc.reset_state()
        train_str_mae.reset_state()
        val_emo_acc.reset_state()
        val_str_mae.reset_state()

        # ── Train ──────────────────────────────────────────────────────────
        epoch_train_loss = 0.0
        n_steps = 0
        for step, (inputs, labels) in enumerate(train_ds):
            t_loss, l_emo, l_str = train_step(inputs, labels, optimizer)
            epoch_train_loss += float(t_loss)
            n_steps += 1

        # ── Validation ─────────────────────────────────────────────────────
        for inputs, labels in val_ds:
            val_step(inputs, labels)

        # ── Collect metrics ────────────────────────────────────────────────
        ep_train_loss   = epoch_train_loss / n_steps
        ep_train_acc    = float(train_emo_acc.result())
        ep_train_mae    = float(train_str_mae.result())
        ep_val_acc      = float(val_emo_acc.result())
        ep_val_mae      = float(val_str_mae.result())
        ep_time         = time.time() - ep_start
        current_lr      = float(sched(optimizer.iterations))

        # ── Simpan ke history ──────────────────────────────────────────────
        history_store['loss'].append(ep_train_loss)
        history_store['emotion_output_acc'].append(ep_train_acc)
        history_store['stress_output_mae'].append(ep_train_mae)
        history_store['val_emotion_output_acc'].append(ep_val_acc)
        history_store['val_stress_output_mae'].append(ep_val_mae)

        # ── Tulis ke TensorBoard ──────────────────────────────────────────
        tb_step = epoch_offset + epoch
        with train_summary_writer.as_default():
            tf.summary.scalar('total_loss', ep_train_loss, step=tb_step)
            tf.summary.scalar('emotion_acc', ep_train_acc, step=tb_step)
            tf.summary.scalar('stress_mae', ep_train_mae, step=tb_step)
            tf.summary.scalar('learning_rate', current_lr, step=tb_step)

        with val_summary_writer.as_default():
            tf.summary.scalar('emotion_acc', ep_val_acc, step=tb_step)
            tf.summary.scalar('stress_mae', ep_val_mae, step=tb_step)

        # ── Log per epoch ──────────────────────────────────────────────────
        print(
            f'Epoch {epoch+1:03d}/{n_epochs} | {ep_time:.1f}s | lr={current_lr:.2e} '
            f'| loss={ep_train_loss:.4f} | emo_acc={ep_train_acc:.4f} '
            f'| str_mae={ep_train_mae:.4f} '
            f'| val_acc={ep_val_acc:.4f} | val_mae={ep_val_mae:.4f}'
        )

        # ── Save best weights ──────────────────────────────────────────────
        if best_ckpt_path and ep_val_acc > best_val_acc:
            best_val_acc = ep_val_acc
            model.save_weights(best_ckpt_path)
            print(f'  Best checkpoint saved (val_acc={ep_val_acc:.4f})')

    total_time = time.time() - train_start
    h, rem = divmod(total_time, 3600)
    m, s   = divmod(rem, 60)
    print('=' * 80)
    print(f'✅ [Vitara AI] {phase_name} Completed in {int(h):02d}:{int(m):02d}:{s:.1f}')
    print('=' * 80)
    return history_store

print('✅ Custom training loop (tf.GradientTape) siap digunakan')

✅ TensorBoard logs will be saved to: /content/drive/MyDrive/vitara/logs/nlp/20260522-074514
✅ Custom training loop (tf.GradientTape) siap digunakan


## 8. Phase 1 — Head Warmup (BERT Frozen)

> Loop manual; BERT di-freeze; hanya head emotion & stress yang dilatih.

In [16]:
# History container untuk Phase 1
history_p1 = {
    'loss': [], 'emotion_output_acc': [], 'stress_output_mae': [],
    'val_emotion_output_acc': [], 'val_stress_output_mae': []
}

history_p1 = run_training_loop(
    phase_name='Phase 1 (BERT Frozen)',
    n_epochs=PHASE1_EPOCHS,
    lr=LR_PHASE1,
    bert_trainable=False,
    history_store=history_p1,
    best_ckpt_path=None,   # Phase 1 tidak perlu save checkpoint
)

print('\n✅ Phase 1 complete')

  BERT trainable=False | Trainable params: 808,326
🚀 [Vitara AI] Phase 1 (BERT Frozen) Started at 2026-05-22 14:45:15 WIB
Epoch 001/5 | 92.8s | lr=2.67e-04 | loss=0.7678 | emo_acc=0.5271 | str_mae=0.2325 | val_acc=0.7555 | val_mae=0.1790
Epoch 002/5 | 81.3s | lr=2.00e-04 | loss=0.4660 | emo_acc=0.7011 | str_mae=0.1859 | val_acc=0.7635 | val_mae=0.1575
Epoch 003/5 | 83.9s | lr=1.33e-04 | loss=0.4067 | emo_acc=0.7322 | str_mae=0.1768 | val_acc=0.7856 | val_mae=0.1526
Epoch 004/5 | 84.2s | lr=6.67e-05 | loss=0.3791 | emo_acc=0.7526 | str_mae=0.1674 | val_acc=0.8016 | val_mae=0.1482
Epoch 005/5 | 83.3s | lr=0.00e+00 | loss=0.3552 | emo_acc=0.7629 | str_mae=0.1651 | val_acc=0.8036 | val_mae=0.1453
✅ [Vitara AI] Phase 1 (BERT Frozen) Completed in 00:07:5.9

✅ Phase 1 complete


## 9. Phase 2 — Full IndoBERT Fine-tune

> BERT di-unfreeze; best checkpoint disimpan berdasarkan `val_emotion_acc` tertinggi.

In [17]:
ckpt_path = os.path.join(GDRIVE_OUT_DIR, 'best_model_weights')

# History container untuk Phase 2
history_p2 = {
    'loss': [], 'emotion_output_acc': [], 'stress_output_mae': [],
    'val_emotion_output_acc': [], 'val_stress_output_mae': []
}

history_p2 = run_training_loop(
    phase_name='Phase 2 (Full Fine-tune)',
    n_epochs=PHASE2_EPOCHS,
    lr=LR_PHASE2,
    bert_trainable=True,
    history_store=history_p2,
    best_ckpt_path=ckpt_path,
    epoch_offset=PHASE1_EPOCHS,
)

print('\n✅ Phase 2 complete')

  BERT trainable=True | Trainable params: 125,249,670
🚀 [Vitara AI] Phase 2 (Full Fine-tune) Started at 2026-05-22 14:52:22 WIB


Epoch 001/15 | 260.4s | lr=1.33e-05 | loss=0.2977 | emo_acc=0.8097 | str_mae=0.1529 | val_acc=0.8310 | val_mae=0.1400
  Best checkpoint saved (val_acc=0.8310)
Epoch 002/15 | 208.7s | lr=1.93e-05 | loss=0.2290 | emo_acc=0.8538 | str_mae=0.1408 | val_acc=0.8517 | val_mae=0.1363
  Best checkpoint saved (val_acc=0.8517)
Epoch 003/15 | 206.9s | lr=1.78e-05 | loss=0.1392 | emo_acc=0.9112 | str_mae=0.1235 | val_acc=0.8383 | val_mae=0.1222
Epoch 004/15 | 205.8s | lr=1.63e-05 | loss=0.0820 | emo_acc=0.9495 | str_mae=0.1093 | val_acc=0.8644 | val_mae=0.1307
  Best checkpoint saved (val_acc=0.8644)
Epoch 005/15 | 206.6s | lr=1.48e-05 | loss=0.0478 | emo_acc=0.9742 | str_mae=0.0983 | val_acc=0.8591 | val_mae=0.1282
Epoch 006/15 | 206.0s | lr=1.33e-05 | loss=0.0239 | emo_acc=0.9874 | str_mae=0.0905 | val_acc=0.8544 | val_mae=0.1197
Epoch 007/15 | 205.3s | lr=1.19e-05 | loss=0.0199 | emo_acc=0.9931 | str_mae=0.0849 | val_acc=0.8564 | val_mae=0.1180
Epoch 008/15 | 205.6s | lr=1.04e-05 | loss=0.0142 |

## 10. Evaluation

> Load best checkpoint dari Phase 2, lalu evaluasi di test set via `model.predict()`.

In [18]:
print('Loading best checkpoint ...')
model.load_weights(ckpt_path)

# Kumpulkan prediksi
emo_preds_list, str_preds_list = [], []
for inputs, _ in test_ds:
    emo_pred, str_pred = model(
        [inputs['input_ids'], inputs['attention_mask'], inputs['token_type_ids']],
        training=False
    )
    emo_preds_list.append(emo_pred.numpy())
    str_preds_list.append(str_pred.numpy())

emo_preds = np.concatenate(emo_preds_list, axis=0)
str_preds = np.concatenate(str_preds_list, axis=0)

emo_pred_labels = np.argmax(emo_preds, axis=1)
final_acc = np.mean(emo_pred_labels == ye_test)
final_mae = np.mean(np.abs(str_preds.flatten() - ys_test))

print('\n' + '='*55)
print(f'  Emotion Accuracy : {final_acc*100:.2f}%   (target ≥ 85%)')
print(f'  Stress MAE       : {final_mae:.4f}    (target ≤ 0.02)')
print('='*55)
print(f'  Accuracy: {"✅" if final_acc >= 0.85 else "❌"} | MAE: {"✅" if final_mae <= 0.02 else "❌"}')
print('='*55)

Loading best checkpoint ...

  Emotion Accuracy : 85.68%   (target ≥ 85%)
  Stress MAE       : 0.1103    (target ≤ 0.02)
  Accuracy: ✅ | MAE: ❌


In [ ]:
print('\nClassification Report:')
print(classification_report(ye_test, emo_pred_labels,
                             target_names=label_encoder.classes_))
print(f'\nStress MAE: {final_mae:.4f}')

## 11. Save ke GDrive

In [20]:
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')

# Full model
model_path = os.path.join(GDRIVE_OUT_DIR, f'vitara_nlp_indobert_{timestamp}.keras')
model.save(model_path)
print(f'✅ Model saved   → {model_path}')

# Tokenizer
tok_path = os.path.join(GDRIVE_OUT_DIR, 'tokenizer')
tokenizer.save_pretrained(tok_path)
print(f'✅ Tokenizer     → {tok_path}')

# Label map
label_map = {str(i): c for i, c in enumerate(label_encoder.classes_.tolist())}
lm_path   = os.path.join(GDRIVE_OUT_DIR, 'label_map.json')
with open(lm_path, 'w') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)
print(f'✅ Label map     → {lm_path}')

# Config
cfg = {
    'model_name': MODEL_NAME, 'max_len': MAX_LEN,
    'num_emotions': NUM_EMOTIONS, 'emotion_labels': EMOTION_LABELS,
    'dropout_rate': DROPOUT_RATE, 'trained_at': timestamp,
    'test_accuracy': round(float(final_acc), 4),
    'test_stress_mae': round(float(final_mae), 4),
    'training_loop': 'tf.GradientTape',
}
cfg_path = os.path.join(GDRIVE_OUT_DIR, 'model_config.json')
with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)
print(f'✅ Config        → {cfg_path}')

print(f'\nAll artifacts saved → {GDRIVE_OUT_DIR}')

/usr/local/lib/python3.12/dist-packages/transformers/generation/tf_utils.py:465: UserWarning: `seed_generator` is deprecated and will be removed in a future version.
  warnings.warn("`seed_generator` is deprecated and will be removed in a future version.", UserWarning)


✅ Model saved   → /content/drive/MyDrive/vitara/models/nlp/vitara_nlp_indobert_20260522_0846.keras
✅ Tokenizer     → /content/drive/MyDrive/vitara/models/nlp/tokenizer
✅ Label map     → /content/drive/MyDrive/vitara/models/nlp/label_map.json
✅ Config        → /content/drive/MyDrive/vitara/models/nlp/model_config.json

All artifacts saved → /content/drive/MyDrive/vitara/models/nlp/


### Export Models (H5, SavedModel, TFLite)
Exporting to multiple formats for deployment flexibility.

In [21]:
import tensorflow as tf

# 1. Export as H5
h5_path = os.path.join(GDRIVE_OUT_DIR, f'vitara_model_{timestamp}.h5')
model.save(h5_path, save_format='h5')
print(f'✅ Model H5 saved      → {h5_path}')

# 2. Export as SavedModel
sm_path = os.path.join(GDRIVE_OUT_DIR, f'vitara_saved_model_{timestamp}')
model.save(sm_path, save_format='tf')
print(f'✅ SavedModel saved     → {sm_path}')

# 3. Export as TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Enable ops for BERT (Select TF Ops)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()
tflite_path = os.path.join(GDRIVE_OUT_DIR, f'vitara_model_{timestamp}.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f'✅ TFLite model saved   → {tflite_path}')

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


✅ Model H5 saved      → /content/drive/MyDrive/vitara/models/nlp/vitara_model_20260522_0846.h5
✅ SavedModel saved     → /content/drive/MyDrive/vitara/models/nlp/vitara_saved_model_20260522_0846
✅ TFLite model saved   → /content/drive/MyDrive/vitara/models/nlp/vitara_model_20260522_0846.tflite


## 12. Training Curves

In [ ]:
import matplotlib.pyplot as plt

# Gabung history Phase 1 + Phase 2
acc     = history_p1['emotion_output_acc']     + history_p2['emotion_output_acc']
val_acc = history_p1['val_emotion_output_acc'] + history_p2['val_emotion_output_acc']
mae     = history_p1['stress_output_mae']      + history_p2['stress_output_mae']
val_mae = history_p1['val_stress_output_mae']  + history_p2['val_stress_output_mae']
ep      = range(1, len(acc)+1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ep, acc,     'o-', label='Train Acc')
axes[0].plot(ep, val_acc, 's--', label='Val Acc')
axes[0].axhline(0.85, color='red', linestyle=':', label='Target 85%')
axes[0].axvline(PHASE1_EPOCHS, color='gray', linestyle='--', alpha=0.5, label='Phase 2 start')
axes[0].set_title('Emotion Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, mae,     'o-', label='Train MAE')
axes[1].plot(ep, val_mae, 's--', label='Val MAE')
axes[1].axhline(0.02, color='red', linestyle=':', label='Target 0.02')
axes[1].axvline(PHASE1_EPOCHS, color='gray', linestyle='--', alpha=0.5, label='Phase 2 start')
axes[1].set_title('Stress MAE'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Vitara AI — IndoBERT NLP (GradientTape)', fontsize=14, fontweight='bold')
plt.tight_layout()
plot_path = os.path.join(GDRIVE_OUT_DIR, f'curves_{timestamp}.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot saved → {plot_path}')

## 13. Inference Demo

In [23]:
def predict(texts_list):
    enc = tokenizer(
        texts_list, max_length=MAX_LEN, padding='max_length',
        truncation=True, return_tensors='tf'
    )
    emo_logits, str_logits = model(
        [enc['input_ids'], enc['attention_mask'], enc['token_type_ids']],
        training=False
    )
    results = []
    for i in range(len(texts_list)):
        emo_idx = int(tf.argmax(emo_logits[i]))
        results.append({
            'emotion':      label_encoder.classes_[emo_idx],
            'stress_level': round(float(str_logits[i][0]), 4)
        })
    return results


samples = [
    'Hari ini aku sangat bahagia, presentasi berjalan lancar!',
    'Deadline besok tapi pekerjaan belum selesai, aku stres banget.',
    'Biasa aja sih, tidak ada yang spesial hari ini.',
    'Aku marah sekali, sudah dua jam menunggu tidak ada kabar!',
    'Khawatir banget soal hasil ujian minggu depan.',
]

print('Inference Demo:\n')
for text, pred in zip(samples, predict(samples)):
    print(f'  {text}')
    print(f'  → {pred}\n')

Inference Demo:

  Hari ini aku sangat bahagia, presentasi berjalan lancar!
  → {'emotion': np.str_('happy'), 'stress_level': 0.1012}

  Deadline besok tapi pekerjaan belum selesai, aku stres banget.
  → {'emotion': np.str_('anxious'), 'stress_level': 0.7419}

  Biasa aja sih, tidak ada yang spesial hari ini.
  → {'emotion': np.str_('neutral'), 'stress_level': 0.248}

  Aku marah sekali, sudah dua jam menunggu tidak ada kabar!
  → {'emotion': np.str_('angry'), 'stress_level': 0.8752}

  Khawatir banget soal hasil ujian minggu depan.
  → {'emotion': np.str_('anxious'), 'stress_level': 0.8442}



---
## Summary

| Item | Value |
|------|-------|
| Base Model | `indobenchmark/indobert-base-p2` |
| BERT Wrapper | `IndoBERTEncoder` (subclass Layer) |
| Training Loop | **`tf.GradientTape` manual** (bukan `model.fit`) |
| Phase 1 | 5 ep, LR=3e-4, BERT frozen |
| Phase 2 | 15 ep, LR=2e-5, full fine-tune |
| Gradient Clip | `tf.clip_by_global_norm(grads, 1.0)` |
| Emotion Loss | FocalLoss (γ=2) + Label Smoothing (0.1) |
| Stress Loss | Huber (δ=0.1) |
| Metrics | Dikelola manual via `tf.keras.metrics` |
| Checkpoint | Disimpan di best `val_emotion_acc` |
| TensorBoard | Terintegrasi manual via `tf.summary` (Log tersimpan di GDrive/Repo) |

> _Vitara AI — NLP Training Notebook_
